In [ ]:
%load_ext autoreload
%autoreload 2

In [2]:
from main import run_experiment, save_results
from simlm.config import (
    Config,
    ExperimentConfig,
    LLMConfig,
    GroundConfig,
)
import numpy as np

In [12]:
experiment_types = ["baseline_cot", "simlm"]
max_iterations = [3,5,7]
models = [
    ("openai", "gpt-3.5-turbo"),
    ("openai", "gpt-4.1-nano"),
    ("google", "gemini-2.0-flash"),
    ("google", "gemini-2.5-flash-preview-04-17"),
    ("google", "gemini-1.5-flash"),
    ("ollama", "gemma3:1b"),
    ("ollama", "gemma3:4b"),
    # ("ollama", "gemma3:12b"),
    # ("ollama", "gemma3:27b"),
    # ("ollama", "llama3.2:1b"),
    # ("ollama", "llama3.2:3b"),
]
flat_grounds = [{"type": "flat"}]
sine_grounds = [{"type": "sine", "amplitude": 0.5, "frequency": 1.0}]
interpolated_grounds = [
    {
        "type": "interpolated",
        "difficulty": 1.0,
        "easy": {"amplitude": 0.15, "frequency": 0.25},
        "hard": {
            "amplitudes": [0.6, 0.15, 0.05],
            "frequencies": [0.9, 2.25, 4.5],
        },
    }
    # for difficulty in np.arange(0.0, 1.1, 0.1)
]
grounds = flat_grounds + sine_grounds + interpolated_grounds

In [15]:
# get the product of experiment types, models, and grounds with
# itertools.product
from itertools import product

configs = []

for experiment_type, (model_service, model_name), ground, max_iteration in product(
    experiment_types,
    models,
    grounds,
    max_iterations
    
):
    config = Config(
        experiment=ExperimentConfig(
            type=experiment_type,
            visualize=False,
            save_results=True,
            max_iterations=max_iteration
        ),
        llm=LLMConfig(
            service=model_service,
            model_name=model_name,
            temperature=0.0,
        ),
        ground=GroundConfig(**ground),
    )
    configs.append(config)
    # print(config)
print(len(configs))

108


In [10]:
for config in configs:
    print(config)
    runner, results = run_experiment(config)
    save_results(results)

experiment=ExperimentConfig(type='baseline_cot', visualize=False, save_results=True, few_shot_examples_path='examples/few_shot_data.yaml', target_distance=50.0, target_bounce_number=3, tolerance=1, max_iterations=5) llm=LLMConfig(service='openai', model_name='gpt-3.5-turbo', temperature=0.0) simulation=SimulationConfig(fps=1000, max_duration=20, gravity=Coordinate(x=0.0, y=-9.81, z=0.0)) projectile=ProjectileConfig(elasticity=0.9, mass=1.0, radius=0.05) ground=GroundConfig(type='flat', friction=0.8, x_min=-200, x_max=400, step=0.1, amplitude=1, frequency=0.5, difficulty=0.9, easy=EasyGroundConfig(amplitude=1, frequency=0.5), hard=HardGroundConfig(amplitudes=[0.6, 0.15, 0.05], frequencies=[0.9, 2.25, 4.5]))

Running Baseline CoT for FlatGround()


2025-04-23 21:55:23,103 - INFO - HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


LLM Proposed: h=15.20m, v=30.50m/s
LLM Reasoning: First, I calculated the time of flight for the projectile to reach the ground after the 3rd bounce. Then, I estimated the horizontal distance covered during each bounce considering the elasticity of the collisions. Finally, I adjusted the initial velocity to achieve a landing position close to 50.0m.
Debug: Starting simulation. Target bounces: 3
Debug: Initial position: Vec2d(0.0, 15.2), velocity: Vec2d(30.5, 0.0)
Debug: Reached target bounces (3) at step 9476.
Debug: Simulation finished. Bounces recorded: 3. Duration: 0.08s
Debug: Final bounce locations (first three): [53.61900000000247, 150.09050000000332, 261.1417486528389]
Result: Bounces=[53.61900000000247, 150.09050000000332, 261.1417486528389], Target Bounce Dist=261.14m, Error=211.14m
Finished CoT in 3.16s
experiment=ExperimentConfig(type='baseline_cot', visualize=False, save_results=True, few_shot_examples_path='examples/few_shot_data.yaml', target_distance=50.0, target_bounce_

2025-04-23 21:55:24,947 - INFO - HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


LLM Proposed: h=10.00m, v=30.00m/s
LLM Reasoning: First, I calculated the period of the sinusoidal ground function to estimate the time between bounces. Then, I adjusted the initial height to account for the sinusoidal shape of the ground. Finally, I fine-tuned the initial velocity to ensure the 3rd bounce lands near 50.0m.
Debug: Starting simulation. Target bounces: 3
Debug: Initial position: Vec2d(0.0, 10.0), velocity: Vec2d(30.0, 0.0)
Debug: Reached target bounces (3) at step 12696.
Debug: Simulation finished. Bounces recorded: 3. Duration: 0.11s
Debug: Final bounce locations (first three): [43.35000000000088, 117.85799989351143, 180.18934641505396]
Result: Bounces=[43.35000000000088, 117.85799989351143, 180.18934641505396], Target Bounce Dist=180.19m, Error=130.19m
Finished CoT in 1.91s
experiment=ExperimentConfig(type='baseline_cot', visualize=False, save_results=True, few_shot_examples_path='examples/few_shot_data.yaml', target_distance=50.0, target_bounce_number=3, tolerance=1, 

...

2025-04-23 22:05:38,788 - ERROR - An error occurred with Ollama: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
2025-04-23 22:05:38,789 - ERROR - Error: Failed to get valid reasoning and parameters from LLM on iteration 1.


experiment=ExperimentConfig(type='simlm', visualize=False, save_results=True, few_shot_examples_path='examples/few_shot_data.yaml', target_distance=50.0, target_bounce_number=3, tolerance=1, max_iterations=5) llm=LLMConfig(service='ollama', model_name='llama3.2:3b', temperature=0.0) simulation=SimulationConfig(fps=1000, max_duration=20, gravity=Coordinate(x=0.0, y=-9.81, z=0.0)) projectile=ProjectileConfig(elasticity=0.9, mass=1.0, radius=0.05) ground=GroundConfig(type='flat', friction=0.8, x_min=-200, x_max=400, step=0.1, amplitude=1, frequency=0.5, difficulty=0.9, easy=EasyGroundConfig(amplitude=1, frequency=0.5), hard=HardGroundConfig(amplitudes=[0.6, 0.15, 0.05], frequencies=[0.9, 2.25, 4.5]))

Running SimLM for FlatGround()

SimLM Iteration 1/5


2025-04-23 22:05:49,781 - INFO - HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 404 Not Found"
2025-04-23 22:05:58,883 - ERROR - An error occurred with Ollama: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
2025-04-23 22:05:58,883 - ERROR - Error: Failed to get valid reasoning and parameters from LLM on iteration 1.


experiment=ExperimentConfig(type='simlm', visualize=False, save_results=True, few_shot_examples_path='examples/few_shot_data.yaml', target_distance=50.0, target_bounce_number=3, tolerance=1, max_iterations=5) llm=LLMConfig(service='ollama', model_name='llama3.2:3b', temperature=0.0) simulation=SimulationConfig(fps=1000, max_duration=20, gravity=Coordinate(x=0.0, y=-9.81, z=0.0)) projectile=ProjectileConfig(elasticity=0.9, mass=1.0, radius=0.05) ground=GroundConfig(type='sine', friction=0.8, x_min=-200, x_max=400, step=0.1, amplitude=0.5, frequency=1.0, difficulty=0.9, easy=EasyGroundConfig(amplitude=1, frequency=0.5), hard=HardGroundConfig(amplitudes=[0.6, 0.15, 0.05], frequencies=[0.9, 2.25, 4.5]))

Running SimLM for SineGround(amplitude=0.5, frequency=1.0)

SimLM Iteration 1/5


2025-04-23 22:06:21,179 - ERROR - An error occurred with Ollama: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
2025-04-23 22:06:21,180 - ERROR - Error: Failed to get valid reasoning and parameters from LLM on iteration 1.


In [9]:
import pandas as pd
from pathlib import Path

results_dir = Path("results")
result_file = results_dir / "experiment_results.jsonl"

result_df = pd.read_json(result_file, lines=True)
config_df = pd.json_normalize(result_df["config"])
result_df = pd.concat([result_df, config_df], axis=1)
result_df = result_df.drop(columns=["config"])
result_df

<string>:1: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
C:\Users\alejandro\AppData\Local\Temp\ipykernel_14528\2220647746.py:7: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  result_df = pd.read_json(result_file, lines=True)


ValueError: Expected object or value